# 5 - Artist-Aware, Genre-Stratified Split (70/15/15)

**Covers:** Section 4.1.1 (split design), Section 4.2.1 (split application), Section 4.9.1 (training-partition integrity, manifest reconciliation).

## Why this notebook is not a two-line `train_test_split`

Section 4.9.1 asserts that *"the intersection of artist identifiers between any two partitions is empty."* Taken literally, collaborations chain artists together transitively. Measured on this corpus, that chaining produces **one connected component holding ~51% of all tracks** -- so a strict component-based split forces that entire component into train and leaves validation and test drawn only from artists outside the mainstream collaboration network. That pool is ~9 popularity points lower on average and 58% Rock against 17% in the component, which is not a train/test split of one population.

**Option C is what this notebook implements:**

1. Group by **primary artist** (`artist_ids[0]`), which keeps groups small and lets genre stratification work.
2. Split 70/15/15 with `StratifiedGroupKFold`, stratifying on genre, grouping on primary artist.
3. **Edge cut:** drop from validation any track sharing *any* artist with train; then drop from test any track sharing any artist with train or validation.
4. Assert the Section 4.9.1 invariant holds exactly, and record every dropped track in the manifest by reason.

The invariant then holds on the literal reading, stratification survives, and the cost is a bounded, reported exclusion rather than a silently skewed test set.

**Produces:** `artifacts/splits.csv`, `artifacts/manifest.json`.

In [ ]:
import json
import os
import sys
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

sys.path.insert(0, os.path.abspath('.'))
import modeling_config as mc

pd.set_option('display.width', 160)

df = mc.load_corpus()
df['artist_list'] = mc.artist_lists(df['artist_ids'])
df['primary_artist'] = df['artist_list'].str[0]
print('rows:', len(df))
print('tracks with 2+ credited artists: %d (%.1f%%)'
      % ((df['artist_list'].str.len() > 1).sum(),
         100 * (df['artist_list'].str.len() > 1).mean()))
print('unique primary artists:', df['primary_artist'].nunique())

## Step 1 - Record the co-appearance diagnostic

This is the measurement that ruled out a strict component-based split. It is recorded in the manifest so the choice of Option C can be justified from the data rather than asserted.

In [ ]:
all_artists = {a for lst in df['artist_list'] for a in lst}
parent = {a: a for a in all_artists}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb

for lst in df['artist_list']:
    for other in lst[1:]:
        union(lst[0], other)

component = df['artist_list'].str[0].map(find)
comp_sizes = component.value_counts()

coappearance = {
    'unique_artists': len(all_artists),
    'multi_artist_tracks': int((df['artist_list'].str.len() > 1).sum()),
    'connected_components': int(len(comp_sizes)),
    'largest_component_tracks': int(comp_sizes.iloc[0]),
    'largest_component_share_pct': float(100 * comp_sizes.iloc[0] / len(df)),
    'unique_primary_artists': int(df['primary_artist'].nunique()),
    'largest_primary_group_share_pct': float(
        100 * df['primary_artist'].value_counts().iloc[0] / len(df)),
}
print(json.dumps(coappearance, indent=2))
print('\nA 15%% test partition cannot contain the largest component'
      ' (%.1f%% of the corpus), which is why grouping is by primary artist'
      ' with an explicit edge cut rather than by component.'
      % coappearance['largest_component_share_pct'])

## Step 2 - Stratified group split, 70/15/15

`StratifiedGroupKFold` with 20 folds at a fixed seed: folds 0-13 to train (70%), 14-16 to validation (15%), 17-19 to test (15%). Groups are primary artists; the stratification label is genre. Section 4.1.1 explicitly subordinates stratification to the grouping constraint -- *"genre proportions are preserved as closely as the artist-grouping constraint allows."*

In [ ]:
N_FOLDS = 20
TRAIN_FOLDS, VAL_FOLDS, TEST_FOLDS = set(range(0, 14)), set(range(14, 17)), set(range(17, 20))

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=mc.RANDOM_SEED)
fold_of = np.empty(len(df), dtype=int)
for fold_idx, (_, holdout) in enumerate(
        sgkf.split(df, y=df['genre'], groups=df['primary_artist'])):
    fold_of[holdout] = fold_idx

df['fold'] = fold_of
df['partition'] = np.select(
    [df['fold'].isin(TRAIN_FOLDS), df['fold'].isin(VAL_FOLDS), df['fold'].isin(TEST_FOLDS)],
    ['train', 'val', 'test'],
    default='unassigned',
)
assert (df['partition'] != 'unassigned').all()

pre_cut = df['partition'].value_counts()
print('before the edge cut:')
print((pre_cut / len(df) * 100).round(2).to_string())
print()
print(pre_cut.to_string())

## Step 3 - The edge cut (Option C)

Train is authoritative and keeps every track. Validation then gives up any track crediting an artist who appears anywhere in train; test gives up any track crediting an artist in train **or** validation. After this, no artist can appear in two partitions under any definition.

In [ ]:
def artists_in(partition):
    sub = df.loc[df['partition'] == partition, 'artist_list']
    return {a for lst in sub for a in lst}

train_artists = artists_in('train')

val_mask = df['partition'] == 'val'
val_conflict = val_mask & df['artist_list'].apply(
    lambda lst: any(a in train_artists for a in lst))
df.loc[val_conflict, 'partition'] = 'dropped_val_shares_artist_with_train'

val_artists = artists_in('val')
blocked = train_artists | val_artists

test_mask = df['partition'] == 'test'
test_conflict = test_mask & df['artist_list'].apply(
    lambda lst: any(a in blocked for a in lst))
df.loc[test_conflict, 'partition'] = 'dropped_test_shares_artist_with_train_or_val'

edge_cut = {
    'val_dropped': int(val_conflict.sum()),
    'val_dropped_pct_of_val': float(100 * val_conflict.sum() / max(1, pre_cut.get('val', 0))),
    'test_dropped': int(test_conflict.sum()),
    'test_dropped_pct_of_test': float(100 * test_conflict.sum() / max(1, pre_cut.get('test', 0))),
}
print(json.dumps(edge_cut, indent=2))
print()
post_cut = df['partition'].value_counts()
print(post_cut.to_string())

## Step 4 - Section 4.9.1 assertions

Training-partition integrity. If any of these fail, nothing downstream is interpretable and the notebook must stop here.

In [ ]:
modelled = df[df['partition'].isin(['train', 'val', 'test'])].copy()

sets = {p: artists_in(p) for p in ['train', 'val', 'test']}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap = sets[a] & sets[b]
    print('artists shared by %-5s and %-5s: %d' % (a, b, len(overlap)))
    assert not overlap, f'artist leakage between {a} and {b}: {list(overlap)[:5]}'

assert modelled['id'].is_unique, 'a track appears in more than one partition'

shares = modelled['partition'].value_counts(normalize=True) * 100
print('\nfinal partition shares (%% of modelled rows):')
print(shares.round(2).to_string())
print('\nSection 4.9.1 training-partition integrity: PASS')

In [ ]:
# Did stratification survive the edge cut, and did the target distribution
# stay comparable across partitions? A large popularity gap between train and
# test would depress R2 for reasons unrelated to any feature.
genre_mix = (pd.crosstab(modelled['genre'], modelled['partition'], normalize='columns') * 100)
genre_mix['corpus'] = df['genre'].value_counts(normalize=True) * 100
print('genre mix, % of each partition:')
print(genre_mix[['corpus', 'train', 'val', 'test']].round(1).to_string())

print('\npopularity by partition:')
print(modelled.groupby('partition')['popularity']
      .agg(['count', 'mean', 'median', 'std']).round(2).to_string())

spread = modelled.groupby('partition')['popularity'].mean()
print('\nmax mean-popularity gap between partitions: %.2f points'
      % (spread.max() - spread.min()))
print('Investigate if this exceeds ~1.5 points -- it biases R2 directly.')

## Step 5 - Write the manifest (Section 4.2, Section 4.1.5)

*"This manifest becomes the single authoritative record for subsequent feature engineering, model training, and reporting, ensuring that every reported result can be retraced to the exact set of tracks that produced it."*

In [ ]:
manifest = {
    'random_seed': mc.RANDOM_SEED,
    'split_ratio_target': {'train': 70, 'val': 15, 'test': 15},
    'split_method': (
        'StratifiedGroupKFold(n_splits=20, shuffle=True) on genre, grouped by '
        'primary artist; folds 0-13 train, 14-16 val, 17-19 test; followed by '
        'the Option C edge cut removing any val/test track sharing an artist '
        'with an earlier partition.'
    ),
    'corpus_rows': int(len(df)),
    'coappearance_diagnostic': coappearance,
    'edge_cut': edge_cut,
    'partition_counts': {k: int(v) for k, v in post_cut.items()},
    'modelled_rows': int(len(modelled)),
    'partition_shares_pct': shares.round(3).to_dict(),
    'popularity_by_partition': modelled.groupby('partition')['popularity']
        .agg(['count', 'mean', 'std']).round(4).to_dict('index'),
    'genre_mix_pct': genre_mix.round(3).to_dict(),
    'exclusion_reasons': {
        'dropped_val_shares_artist_with_train': edge_cut['val_dropped'],
        'dropped_test_shares_artist_with_train_or_val': edge_cut['test_dropped'],
    },
}
print(mc.save_json(manifest, 'manifest.json'))

splits = df[['id', 'partition', 'primary_artist', 'genre', 'year', 'popularity', 'fold']]
splits_path = mc.artifact('splits.csv')
splits.to_csv(splits_path, index=False)
print(splits_path)
print('\nEvery downstream notebook joins on `id` and filters on `partition`.')

---
### Before moving on

- The three artist-intersection assertions must pass. They are the Section 4.9.1 check, not a formality.
- Check the genre mix table: train/val/test columns should closely track the `corpus` column. If the edge cut has visibly skewed one genre, say so in the results rather than leaving it implicit.
- Check the popularity means. A large train-vs-test gap depresses R2 on its own and would make the Table 11 diagnostic (`Control R2 < 0.05 -> investigate`) fire for the wrong reason.
- The edge-cut counts belong in the Chapter 4 exclusion reporting alongside the Section 4.2 exclusions.

Next: `6_tuning.ipynb`.